# Prompt Evaluation Methods

| Method | Description | Use Cases |
|--------|-------------|-----------|
| **Model Based Grading** | Ask an LLM to analyze the results and score or compare multiple versions | Response quality, Quality of instructions, Completeness of results, Helpfulness, Safety |
| **Code Based Grading** | Programmatically evaluate the results | Checking output length, Validity of structured results, Syntax validation, Readability scores |

> Note: 
> - To have better results on grading, provide more context to the grader on how a good solution looks like.
> - While `task` in the generated dataset is enough for understanding the responses, we've additionally added `format` to assist code-based grader, and `solution_criteria` to assist model-based grader.


In [50]:
# Install dependencies
%pip install anthropic python-dotenv

# Load environment variables from .env file
from dotenv import load_dotenv
load_dotenv()

# Create API client
from anthropic import Anthropic

import json

client = Anthropic()
model = "claude-haiku-4-5"

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [51]:
def add_user_message(messages, text):
    messages.append({"role": "user", "content": text})

def add_assistant_message(messages, text):
    messages.append({"role": "assistant", "content": text})

def chat(messages, system=None, temperature=0.5, stop_sequences=None):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
    }
    if system:
        params["system"] = system
    if stop_sequences:
        params["stop_sequences"] = stop_sequences

    response = client.messages.create(**params)
    return response.content[0].text


## Grading Strategies

### Model-based

In [52]:
# Model-based grading workflow

def grade_by_model(test_case, response):
    
    eval_prompt = f'''
You are an expert AWS code reviewer. Your task is to evaluate the following AI-generated solution.

Original Task:
<task>
{test_case["task"]}
</task>

Solution to Evaluate:
<solution>
{response}
</solution>

Criteria you should use to evaluate the solution:
<criteria>
{test_case["solution_criteria"]}
</criteria>

Output Format
Provide your evaluation as a structured JSON object with the following fields, in this specific order:
- "strengths": An array of 1-3 key strengths
- "weaknesses": An array of 1-3 key areas for improvement
- "reasoning": A concise explanation of your overall assessment
- "score": A number between 1-10

Respond with JSON. Keep your response concise and direct.
Example response shape:
{{
    "strengths": string[],
    "weaknesses": string[],
    "reasoning": string,
    "score": number
}}
    '''

    messages = []
    add_user_message(messages, eval_prompt)
    add_assistant_message(messages, "```json") # Prefilled assistant response
    
    eval_response = chat(messages, stop_sequences=["```"]) # eval_response will be a JSON output
    
    return json.loads(eval_response)

### Code-based

In [53]:
import ast, re

def validate_json(text):
    try:
        json.loads(text.strip())
        return 10 # Score
    except json.JSONDecodeError:
        return 0


def validate_python(text):
    try:
        ast.parse(text.strip())
        return 10
    except SyntaxError:
        return 0


def validate_regex(text):
    try:
        re.compile(text.strip())
        return 10
    except re.error:
        return 0


def grade_by_syntax(test_case, response):
    format = test_case["format"]
    if format == "json":
        return validate_json(response)
    elif format == "python":
        return validate_python(response)
    elif format == "regex":
        return validate_regex(response)
    else:
        raise ValueError(f"Unknown format: {format}")

In [54]:
def run_prompt(test_case):
    '''
    Merges the prompt & test case input, and returns the output.
    '''
    test_prompt = f'''
Please solve the following task:
{test_case["task"]}

- Respond only in one of: Python/JSON/Regex
- Do no add any explanations, or comments to it, and only provide the solution in the specified format.
    ''' # This is the PROMPT WE ARE EVALUATING, which will be used in the evaluation workflow to generate the output for each test case.

    messages = []
    add_user_message(messages, test_prompt)
    add_assistant_message(messages, "```code") # IMPORTANT: Note the use of `code` to cover all of: python/json/bash(regex)
    
    output = chat(messages, stop_sequences=["```"])
    
    return output

In [55]:
def run_test_case(test_case):
    '''
    Runs the test case by calling the run_prompt function and comparing the output to the expected output.
    '''
    output = run_prompt(test_case)

    # Model Grading
    model_grade = grade_by_model(test_case, output)
    code_grade = grade_by_syntax(test_case, output)

    # Extract important evaluation details from model_grade & code_grade
    model_score = model_grade["score"]
    syntax_score = code_grade # returns 10 if syntax is correct, 0 if syntax is incorrect
    reasoning = model_grade["reasoning"]

    average_score = (model_score + syntax_score) / 2
    
    return {
        "output": output,
        "test_case": test_case,
        "score": average_score,
        "reasoning": reasoning,
    }


In [56]:
from statistics import mean


def run_eval(dataset):
    '''
    Runs the evaluation by iterating through the dataset and running each test case by calling the run_test_case function.
    '''
    results = []

    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)

    average_score = mean([result["score"] for result in results])
    print(f"Average Score: {average_score}")
    
    return results

In [57]:
# Read the dataset from the JSON file

with open("evaluation_dataset.json", "r") as f:
    dataset = json.load(f)

results = run_eval(dataset)
print(json.dumps(results, indent=2))

Average Score: 7.833333333333333
[
  {
    "output": "\nimport re\n\ndef parse_s3_uri(uri):\n    pattern = r's3://([a-z0-9\\-\\.]+)(?:/(.*))?'\n    match = re.match(pattern, uri)\n    \n    if match:\n        bucket_name = match.group(1)\n        path = match.group(2) or \"\"\n        return {\n            \"bucket\": bucket_name,\n            \"path\": path\n        }\n    return None\n",
    "test_case": {
      "task": "Parse an AWS S3 bucket name and region from a full S3 URI (e.g., 's3://my-bucket-name/path/to/object') and return them as separate values",
      "format": "regex",
      "solution_criteria": "The regex should correctly extract the bucket name and optional region from various S3 URI formats, handling edge cases like bucket names with hyphens and numbers"
    },
    "score": 7.0,
    "reasoning": "The solution demonstrates solid regex fundamentals and handles the basic parsing task competently. However, it fundamentally misinterprets the requirements by extracting the